# 03. Coding agent readiness와 build shaping

목표: agent에게 맡길 task의 spec·verifier·권한을 점검하고, product opportunity를 impact·learning value·risk로 우선순위화합니다.

In [ ]:
def agent_readiness(task: dict) -> tuple[bool, list[str]]:
    checks = {
        "goal defined": bool(task.get("goal")),
        "scope bounded": bool(task.get("allowed_paths")),
        "verifier available": bool(task.get("verifiers")),
        "rollback possible": bool(task.get("rollback")),
        "production writes blocked": not task.get("production_write", False),
    }
    failures = [name for name, passed in checks.items() if not passed]
    return not failures, failures

tasks = [
    {"name": "docs links", "goal": "fix broken links", "allowed_paths": ["docs/"], "verifiers": ["link-check"], "rollback": "git revert"},
    {"name": "prod migration", "goal": "migrate users", "allowed_paths": [], "verifiers": [], "rollback": "", "production_write": True},
]
for task in tasks:
    ready, failures = agent_readiness(task)
    print(task["name"], "READY" if ready else "HUMAN DESIGN NEEDED", failures)

준비되지 않은 task를 더 긴 prompt로 밀어붙이지 마세요. 먼저 scope, verifier, rollback과 authority를 사람이 설계해야 합니다.

In [ ]:
opportunities = [
    {"name": "support answer assistant", "impact": 8, "confidence": 0.7, "effort": 4, "learning": 8, "risk": 3},
    {"name": "autonomous refund approval", "impact": 10, "confidence": 0.4, "effort": 5, "learning": 6, "risk": 10},
    {"name": "ticket summarization", "impact": 6, "confidence": 0.9, "effort": 2, "learning": 7, "risk": 2},
]

for item in opportunities:
    # 학습 가치를 포함하고 위험을 penalty로 반영한 교육용 점수입니다.
    item["score"] = (item["impact"] * item["confidence"] + 0.5 * item["learning"] - item["risk"]) / item["effort"]

for item in sorted(opportunities, key=lambda x: x["score"], reverse=True):
    print(f'{item["score"]:5.2f}  {item["name"]}')

## 해석

점수가 product decision을 대신하지는 않습니다. 후보별 가장 위험한 assumption, 사용자 segment, success/guardrail metric과 중단 조건을 문서화한 뒤 작은 experiment를 설계하세요. high-risk action은 human approval과 policy enforcement가 있어야 합니다.